# CNN–BiLSTM–Attention Model (Wisconsin Breast Cancer Dataset)

### Clone Repo

In [ ]:
!git clone https://github.com/kjs-design/CNN-BiLSTM-Attention-model.git

In [6]:
!ls CNN-BiLSTM-Attention-model
!find CNN-BiLSTM-Attention-model -maxdepth 2 -type f

data            kegg            Model           README.md       wordVectorTrain
CNN-BiLSTM-Attention-model/kegg/true_negative_sample_entry_gene_mapping.txt
CNN-BiLSTM-Attention-model/kegg/false_negative_sample_entry_gene_mapping.xlsx
CNN-BiLSTM-Attention-model/kegg/false_positive_sample_entry_gene_mapping.txt
CNN-BiLSTM-Attention-model/kegg/tp_fn_dot_kegg.png
CNN-BiLSTM-Attention-model/kegg/introduce
CNN-BiLSTM-Attention-model/kegg/true_positive_sample_entry_gene_mapping.xlsx
CNN-BiLSTM-Attention-model/kegg/true_negative_sample_entry_gene_mapping.xlsx
CNN-BiLSTM-Attention-model/kegg/tp_dot_kegg.png
CNN-BiLSTM-Attention-model/kegg/true_positive_sample_entry.txt
CNN-BiLSTM-Attention-model/kegg/fn_dot_kegg.png
CNN-BiLSTM-Attention-model/kegg/false_positive_sample_entry.txt
CNN-BiLSTM-Attention-model/kegg/false_negative_sample_entry.txt
CNN-BiLSTM-Attention-model/kegg/false_positive_sample_entry_gene_mapping.xlsx
CNN-BiLSTM-Attention-model/kegg/false_negative_sample_entry_gene_mapping.txt


### Imports

In [12]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    cohen_kappa_score,
    recall_score,
    precision_score,
    roc_auc_score
)

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, Bidirectional, LSTM,
    Dense, Dropout, GlobalAveragePooling1D, Multiply,
    Softmax, Permute, Lambda
)
from tensorflow.keras.callbacks import EarlyStopping

### Load and Split the Breast Cancer Data

In [13]:
data = load_breast_cancer()
X = data.data
y = data.target

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42
)
# 60% train, 20% val, 20% test

### Scale and Reshape the Data

In [14]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

n_features = X_train_scaled.shape[1]

X_train_seq = X_train_scaled.reshape((X_train_scaled.shape[0], n_features, 1))
X_val_seq   = X_val_scaled.reshape((X_val_scaled.shape[0], n_features, 1))
X_test_seq  = X_test_scaled.reshape((X_test_scaled.shape[0], n_features, 1))

print(X_train_seq.shape, X_val_seq.shape, X_test_seq.shape)

(341, 30, 1) (114, 30, 1) (114, 30, 1)


### Build the CNN–BiLSTM–Attention Model

In [15]:
tf.keras.utils.set_random_seed(42)

inputs = Input(shape=(n_features, 1))

# CNN block
x = Conv1D(filters=32, kernel_size=3, activation="relu", padding="same")(inputs)
x = MaxPooling1D(pool_size=2)(x)

# BiLSTM block
x = Bidirectional(LSTM(32, return_sequences=True))(x)

# Attention block
attention_scores = Dense(1, activation="tanh")(x)       # (batch, time, 1)
attention_scores = tf.keras.layers.Flatten()(attention_scores)   # (batch, time)
attention_weights = Softmax(name="attention_weights")(attention_scores)  # (batch, time)

# Expand weights back to match sequence dimension
attention_weights_expanded = Lambda(lambda z: tf.expand_dims(z, axis=-1))(attention_weights)

# Weighted sequence
x = Multiply()([x, attention_weights_expanded])

# Pool and classify
x = GlobalAveragePooling1D()(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation="sigmoid")(x)

cnn_bilstm_att_model = Model(inputs, outputs)

cnn_bilstm_att_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

cnn_bilstm_att_model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 30, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 30, 32)    │        128 │ input_layer_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 15, 32)    │          0 │ conv1d_2[0][0]    │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_2     │ (None, 15, 64)    │     16,640 │ max_pooling1d_2[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 15, 1)     │         65 │ bidirectional_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 15)        │          0 │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_weights   │ (None, 15)        │          0 │ flatten_1[0][0]   │
│ (Softmax)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 15, 1)     │          0 │ attention_weight… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 15, 64)    │          0 │ bidirectional_2[… │
│                     │                   │            │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ multiply[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 64)        │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 1)         │         65 │ dropout_6[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 16,898 (66.01 KB)

 Trainable params: 16,898 (66.01 KB)

 Non-trainable params: 0 (0.00 B)

### Train the Model

In [16]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history = cnn_bilstm_att_model.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=60,
    batch_size=32,
    verbose=1,
    callbacks=[early_stop]
)

Epoch 1/60
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 35ms/step - accuracy: 0.8211 - auc: 0.8301 - loss: 0.6857 - val_accuracy: 0.7895 - val_auc: 0.9329 - val_loss: 0.6791
Epoch 2/60
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8387 - auc: 0.9401 - loss: 0.6685 - val_accuracy: 0.8070 - val_auc: 0.9471 - val_loss: 0.6578
Epoch 3/60
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8534 - auc: 0.9526 - loss: 0.6379 - val_accuracy: 0.8246 - val_auc: 0.9627 - val_loss: 0.6231
Epoch 4/60
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8856 - auc: 0.9614 - loss: 0.5961 - val_accuracy: 0.8596 - val_auc: 0.9707 - val_loss: 0.5833
Epoch 5/60
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9062 - auc: 0.9681 - loss: 0.5525 - val_accuracy: 0.8860 - val_auc: 0.9774 - val_loss: 0.5436
Epoch 6/60
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9267 - auc: 0.9727 - loss: 0.5181 - val_accuracy: 0.8860 - val_auc: 0.9859 - val_loss: 0.5098
Epoch 7/60
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step -

### Evaluate the Model

In [17]:
# Training and validation accuracy from evaluate()
train_loss, train_acc, train_auc = cnn_bilstm_att_model.evaluate(X_train_seq, y_train, verbose=0)
val_loss, val_acc, val_auc = cnn_bilstm_att_model.evaluate(X_val_seq, y_val, verbose=0)

# Test predictions
y_test_prob = cnn_bilstm_att_model.predict(X_test_seq, verbose=0).ravel()
y_test_pred = (y_test_prob >= 0.5).astype(int)

test_acc = accuracy_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)
kappa = cohen_kappa_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
auc = roc_auc_score(y_test, y_test_prob)

cnn_bilstm_att_results = pd.DataFrame([{
    "Model": "CNN-BiLSTM-Attention",
    "Train_accuracy": round(train_acc, 4),
    "Val_accuracy": round(val_acc, 4),
    "Test_accuracy": round(test_acc, 4),
    "F1_score": round(f1, 4),
    "Kappa": round(kappa, 4),
    "Recall": round(recall, 4),
    "Precision": round(precision, 4),
    "AUC": round(auc, 4)
}])

print(cnn_bilstm_att_results.to_string(index=False))

               Model  Train_accuracy  Val_accuracy  Test_accuracy  F1_score  Kappa  Recall  Precision    AUC
CNN-BiLSTM-Attention          0.9589        0.9649         0.9386    0.9517 0.8674  0.9583     0.9452 0.9788
